In [69]:
import pandas as pd
import mysql.connector
from mlxtend.frequent_patterns import apriori, association_rules

In [71]:
conn = mysql.connector.connect(
    host="127.0.0.1",
    port=3307,
    user="root",
    password="kunmuradstsv12",
    database="retail_project"
)


query = """
SELECT i.bill_no, p.item_name
FROM invoicedetail d
JOIN product p ON d.product_id = p.product_id
JOIN invoice i ON d.bill_no = i.bill_no
"""

df = pd.read_sql(query, conn)

print("Data loaded:", df.shape)

basket = pd.crosstab(df["bill_no"], df["item_name"])

basket = basket.applymap(lambda x: 1 if x > 0 else 0)

print("Basket shape:", basket.shape)

# chạy Apriori
frequent_itemsets = apriori(
    basket,
    min_support=0.02,
    use_colnames=True
)

print("Frequent itemsets:", frequent_itemsets.shape)

# tạo association rules
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.3
)

# sắp xếp theo lift
rules = rules.sort_values("lift", ascending=False)

print("Top rules:")
print(rules[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift"
]].head(10))

C:\Users\ASUS\AppData\Local\Temp\ipykernel_16292\893464035.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Data loaded: (517424, 2)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_16292\893464035.py:23: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket = basket.applymap(lambda x: 1 if x > 0 else 0)


Basket shape: (19385, 3999)


c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


Frequent itemsets: (389, 2)
Top rules:
                                           antecedents  \
149                   (PINK REGENCY TEACUP AND SAUCER)   
144  (GREEN REGENCY TEACUP AND SAUCER, ROSES REGENC...   
147                  (GREEN REGENCY TEACUP AND SAUCER)   
146  (ROSES REGENCY TEACUP AND SAUCER, PINK REGENCY...   
25                    (PINK REGENCY TEACUP AND SAUCER)   
24                   (GREEN REGENCY TEACUP AND SAUCER)   
145  (GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY...   
148                  (ROSES REGENCY TEACUP AND SAUCER)   
22                  (GARDENERS KNEELING PAD KEEP CALM)   
23                 (GARDENERS KNEELING PAD CUP OF TEA)   

                                           consequents   support  confidence  \
149  (GREEN REGENCY TEACUP AND SAUCER, ROSES REGENC...  0.026515    0.699320   
144                   (PINK REGENCY TEACUP AND SAUCER)  0.026515    0.704110   
147  (ROSES REGENCY TEACUP AND SAUCER, PINK REGENCY...  0.026515    0.528263   
14

In [72]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

rules = rules.round(4)

rules.to_csv("association_rules_clean.csv", index=False)

In [73]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(60 TEATIME FAIRY CAKE CASES),(PACK OF 72 RETROSPOT CAKE CASES),0.0412,0.0660,0.0224,0.5451,8.2619,1.0,0.0197,2.0533,0.9167,0.2649,0.5130,0.4426
1,(PACK OF 72 RETROSPOT CAKE CASES),(60 TEATIME FAIRY CAKE CASES),0.0660,0.0412,0.0224,0.3401,8.2619,1.0,0.0197,1.4530,0.9411,0.2649,0.3118,0.4426
2,(ALARM CLOCK BAKELIKE GREEN),(ALARM CLOCK BAKELIKE PINK),0.0499,0.0392,0.0212,0.4250,10.8552,1.0,0.0192,1.6711,0.9555,0.3125,0.4016,0.4833
3,(ALARM CLOCK BAKELIKE PINK),(ALARM CLOCK BAKELIKE GREEN),0.0392,0.0499,0.0212,0.5415,10.8552,1.0,0.0192,2.0722,0.9449,0.3125,0.5174,0.4833
4,(ALARM CLOCK BAKELIKE GREEN),(ALARM CLOCK BAKELIKE RED),0.0499,0.0530,0.0327,0.6546,12.3558,1.0,0.0300,2.7418,0.9673,0.4651,0.6353,0.6355
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181,"(JUMBO SHOPPER VINTAGE RED PAISLEY, JUMBO STOR...",(JUMBO BAG RED RETROSPOT),0.0268,0.1063,0.0200,0.7462,7.0180,1.0,0.0172,3.5206,0.8811,0.1769,0.7160,0.4672
182,"(JUMBO BAG RED RETROSPOT, JUMBO STORAGE BAG SUKI)",(JUMBO SHOPPER VINTAGE RED PAISLEY),0.0372,0.0601,0.0200,0.5381,8.9467,1.0,0.0178,2.0349,0.9225,0.2588,0.5086,0.4355
183,(JUMBO SHOPPER VINTAGE RED PAISLEY),"(JUMBO BAG RED RETROSPOT, JUMBO STORAGE BAG SUKI)",0.0601,0.0372,0.0200,0.3328,8.9467,1.0,0.0178,1.4430,0.9451,0.2588,0.3070,0.4355
184,(JUMBO BAG RED RETROSPOT),"(JUMBO SHOPPER VINTAGE RED PAISLEY, JUMBO STOR...",0.1063,0.0268,0.0200,0.1883,7.0180,1.0,0.0172,1.1989,0.9595,0.1769,0.1659,0.4672


In [74]:
query = """
SELECT 
    p.item_name,
    SUM(d.quantity * d.price) AS revenue
FROM invoicedetail d
JOIN product p
    ON d.product_id = p.product_id
GROUP BY p.item_name
ORDER BY revenue DESC
LIMIT 20
"""

df = pd.read_sql(query, conn)

print(df)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_16292\3922540395.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


                             item_name    revenue
0             REGENCY CAKESTAND 3 TIER  165689.19
1   WHITE HANGING HEART T-LIGHT HOLDER  102588.37
2                        PARTY BUNTING   97367.48
3              JUMBO BAG RED RETROSPOT   93416.03
4                   RABBIT NIGHT LIGHT   66452.11
5       PAPER CHAIN KIT 50'S CHRISTMAS   63458.59
6        ASSORTED COLOUR BIRD ORNAMENT   58459.49
7                        CHILLI LIGHTS   53471.36
8              JUMBO BAG PINK POLKADOT   41829.94
9             BLACK RECORD COVER FRAME   40637.13
10                      SPOTTY BUNTING   40299.08
11      PICNIC BASKET WICKER 60 PIECES   39619.50
12       DOORMAT KEEP CALM AND COME IN   37952.44
13    SET OF 3 CAKE TINS PANTRY DESIGN   35965.09
14   WOOD BLACK BOARD ANT WHITE FINISH   35420.52
15             LUNCH BAG RED RETROSPOT   34866.56
16            JAM MAKING SET WITH JARS   33518.38
17                      POPCORN HOLDER   33377.87
18     VICTORIAN GLASS HANGING T-LIGHT   32874.47


In [80]:
top10_lift = rules.sort_values(by="lift", ascending=False).head(10)

# in ra
top10_lift

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
173,(PINK REGENCY TEACUP AND SAUCER),"(GREEN REGENCY TEACUP AND SAUCER, ROSES REGENC...",0.0379,0.0377,0.0265,0.6993,18.5703,1.0,0.0251,3.2005,0.9834,0.5405,0.6876,0.7017
168,"(GREEN REGENCY TEACUP AND SAUCER, ROSES REGENC...",(PINK REGENCY TEACUP AND SAUCER),0.0377,0.0379,0.0265,0.7041,18.5703,1.0,0.0251,3.2515,0.9832,0.5405,0.6924,0.7017
171,(GREEN REGENCY TEACUP AND SAUCER),"(ROSES REGENCY TEACUP AND SAUCER, PINK REGENCY...",0.0502,0.0294,0.0265,0.5283,17.9972,1.0,0.0250,2.0576,0.9943,0.5000,0.5140,0.7158
170,"(ROSES REGENCY TEACUP AND SAUCER, PINK REGENCY...",(GREEN REGENCY TEACUP AND SAUCER),0.0294,0.0502,0.0265,0.9033,17.9972,1.0,0.0250,9.8262,0.9730,0.5000,0.8982,0.7158
24,(GREEN REGENCY TEACUP AND SAUCER),(PINK REGENCY TEACUP AND SAUCER),0.0502,0.0379,0.0312,0.6208,16.3720,1.0,0.0293,2.5369,0.9885,0.5471,0.6058,0.7213
25,(PINK REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.0379,0.0502,0.0312,0.8218,16.3720,1.0,0.0293,5.3291,0.9759,0.5471,0.8123,0.7213
169,"(GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY...",(ROSES REGENCY TEACUP AND SAUCER),0.0312,0.0522,0.0265,0.8510,16.3009,1.0,0.0249,6.3608,0.9688,0.4664,0.8428,0.6794
172,(ROSES REGENCY TEACUP AND SAUCER),"(GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY...",0.0522,0.0312,0.0265,0.5079,16.3009,1.0,0.0249,1.9688,0.9904,0.4664,0.4921,0.6794
22,(GARDENERS KNEELING PAD KEEP CALM),(GARDENERS KNEELING PAD CUP OF TEA),0.0467,0.0389,0.0281,0.6011,15.4541,1.0,0.0262,2.4094,0.9811,0.4879,0.5850,0.6613
23,(GARDENERS KNEELING PAD CUP OF TEA),(GARDENERS KNEELING PAD KEEP CALM),0.0389,0.0467,0.0281,0.7215,15.4541,1.0,0.0262,3.4229,0.9731,0.4879,0.7078,0.6613


In [81]:
rules 

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(60 TEATIME FAIRY CAKE CASES),(PACK OF 72 RETROSPOT CAKE CASES),0.0412,0.0660,0.0224,0.5451,8.2619,1.0,0.0197,2.0533,0.9167,0.2649,0.5130,0.4426
1,(PACK OF 72 RETROSPOT CAKE CASES),(60 TEATIME FAIRY CAKE CASES),0.0660,0.0412,0.0224,0.3401,8.2619,1.0,0.0197,1.4530,0.9411,0.2649,0.3118,0.4426
2,(ALARM CLOCK BAKELIKE GREEN),(ALARM CLOCK BAKELIKE PINK),0.0499,0.0392,0.0212,0.4250,10.8552,1.0,0.0192,1.6711,0.9555,0.3125,0.4016,0.4833
3,(ALARM CLOCK BAKELIKE PINK),(ALARM CLOCK BAKELIKE GREEN),0.0392,0.0499,0.0212,0.5415,10.8552,1.0,0.0192,2.0722,0.9449,0.3125,0.5174,0.4833
4,(ALARM CLOCK BAKELIKE GREEN),(ALARM CLOCK BAKELIKE RED),0.0499,0.0530,0.0327,0.6546,12.3558,1.0,0.0300,2.7418,0.9673,0.4651,0.6353,0.6355
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181,"(JUMBO SHOPPER VINTAGE RED PAISLEY, JUMBO STOR...",(JUMBO BAG RED RETROSPOT),0.0268,0.1063,0.0200,0.7462,7.0180,1.0,0.0172,3.5206,0.8811,0.1769,0.7160,0.4672
182,"(JUMBO BAG RED RETROSPOT, JUMBO STORAGE BAG SUKI)",(JUMBO SHOPPER VINTAGE RED PAISLEY),0.0372,0.0601,0.0200,0.5381,8.9467,1.0,0.0178,2.0349,0.9225,0.2588,0.5086,0.4355
183,(JUMBO SHOPPER VINTAGE RED PAISLEY),"(JUMBO BAG RED RETROSPOT, JUMBO STORAGE BAG SUKI)",0.0601,0.0372,0.0200,0.3328,8.9467,1.0,0.0178,1.4430,0.9451,0.2588,0.3070,0.4355
184,(JUMBO BAG RED RETROSPOT),"(JUMBO SHOPPER VINTAGE RED PAISLEY, JUMBO STOR...",0.1063,0.0268,0.0200,0.1883,7.0180,1.0,0.0172,1.1989,0.9595,0.1769,0.1659,0.4672


In [83]:
query = """
SELECT 
    p.item_name,
    SUM(d.quantity) AS total_quantity
FROM invoicedetail d
JOIN product p
    ON d.product_id = p.product_id
GROUP BY p.item_name
ORDER BY total_quantity DESC
LIMIT 10
"""

top_products = pd.read_sql(query, conn)
top10 = top_products["item_name"].tolist()

print("Top 10 products:")
print(top10)

# 3 lọc rule theo từng sản phẩm
rules["antecedents"] = rules["antecedents"].astype(str)
rules["consequents"] = rules["consequents"].astype(str)
result = []

for product in top10:

    related_rules = rules[
        rules["antecedents"].str.contains(product) |
        rules["consequents"].str.contains(product)
    ]

    top_rules = related_rules.sort_values("lift", ascending=False)

    # thêm cột main_product
    top_rules["main_product"] = product

    # đưa cột lên đầu
    cols = ["main_product"] + [col for col in top_rules.columns if col != "main_product"]
    top_rules = top_rules[cols]

    result.append(top_rules)

final_rules = pd.concat(result)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_16292\2823511783.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  top_products = pd.read_sql(query, conn)


Top 10 products:
['WORLD WAR 2 GLIDERS ASSTD DESIGNS', 'JUMBO BAG RED RETROSPOT', 'WHITE HANGING HEART T-LIGHT HOLDER', 'ASSORTED COLOUR BIRD ORNAMENT', 'POPCORN HOLDER', 'PACK OF 72 RETROSPOT CAKE CASES', 'RABBIT NIGHT LIGHT', 'PACK OF 12 LONDON TISSUES', 'MINI PAINT SET VINTAGE', 'VICTORIAN GLASS HANGING T-LIGHT']


In [85]:
final_rules.head()

,main_product,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
185,JUMBO BAG RED RETROSPOT,frozenset({'JUMBO STORAGE BAG SUKI'}),frozenset({'JUMBO SHOPPER VINTAGE RED PAISLEY'...,0.0608,0.0349,0.0200,0.3294,9.4451,1.0,0.0179,1.4391,0.9520,0.2647,0.3051,0.4517
180,JUMBO BAG RED RETROSPOT,frozenset({'JUMBO SHOPPER VINTAGE RED PAISLEY'...,frozenset({'JUMBO STORAGE BAG SUKI'}),0.0349,0.0608,0.0200,0.5740,9.4451,1.0,0.0179,2.2046,0.9264,0.2647,0.5464,0.4517
176,JUMBO BAG RED RETROSPOT,"frozenset({'JUMBO BAG RED RETROSPOT', 'JUMBO S...",frozenset({'JUMBO BAG PINK POLKADOT'}),0.0372,0.0625,0.0213,0.5714,9.1471,1.0,0.0189,2.1876,0.9251,0.2711,0.5429,0.4558
177,JUMBO BAG RED RETROSPOT,frozenset({'JUMBO BAG PINK POLKADOT'}),"frozenset({'JUMBO BAG RED RETROSPOT', 'JUMBO S...",0.0625,0.0372,0.0213,0.3402,9.1471,1.0,0.0189,1.4593,0.9500,0.2711,0.3147,0.4558
183,JUMBO BAG RED RETROSPOT,frozenset({'JUMBO SHOPPER VINTAGE RED PAISLEY'}),"frozenset({'JUMBO BAG RED RETROSPOT', 'JUMBO S...",0.0601,0.0372,0.0200,0.3328,8.9467,1.0,0.0178,1.4430,0.9451,0.2588,0.3070,0.4355


In [86]:
# bỏ frozenset
final_rules["antecedents"] = final_rules["antecedents"].str.replace("frozenset({", "", regex=False)
final_rules["antecedents"] = final_rules["antecedents"].str.replace("})", "", regex=False)

final_rules["consequents"] = final_rules["consequents"].str.replace("frozenset({", "", regex=False)
final_rules["consequents"] = final_rules["consequents"].str.replace("})", "", regex=False)

# bỏ dấu nháy
final_rules["antecedents"] = final_rules["antecedents"].str.replace("'", "", regex=False)
final_rules["consequents"] = final_rules["consequents"].str.replace("'", "", regex=False)

# đổi dấu phẩy thành &
final_rules["antecedents"] = final_rules["antecedents"].str.replace(",", " &", regex=False)
final_rules["consequents"] = final_rules["consequents"].str.replace(",", " &", regex=False)

final_rules.head()

,main_product,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
185,JUMBO BAG RED RETROSPOT,JUMBO STORAGE BAG SUKI,JUMBO SHOPPER VINTAGE RED PAISLEY & JUMBO BAG ...,0.0608,0.0349,0.0200,0.3294,9.4451,1.0,0.0179,1.4391,0.9520,0.2647,0.3051,0.4517
180,JUMBO BAG RED RETROSPOT,JUMBO SHOPPER VINTAGE RED PAISLEY & JUMBO BAG ...,JUMBO STORAGE BAG SUKI,0.0349,0.0608,0.0200,0.5740,9.4451,1.0,0.0179,2.2046,0.9264,0.2647,0.5464,0.4517
176,JUMBO BAG RED RETROSPOT,JUMBO BAG RED RETROSPOT & JUMBO STORAGE BAG SUKI,JUMBO BAG PINK POLKADOT,0.0372,0.0625,0.0213,0.5714,9.1471,1.0,0.0189,2.1876,0.9251,0.2711,0.5429,0.4558
177,JUMBO BAG RED RETROSPOT,JUMBO BAG PINK POLKADOT,JUMBO BAG RED RETROSPOT & JUMBO STORAGE BAG SUKI,0.0625,0.0372,0.0213,0.3402,9.1471,1.0,0.0189,1.4593,0.9500,0.2711,0.3147,0.4558
183,JUMBO BAG RED RETROSPOT,JUMBO SHOPPER VINTAGE RED PAISLEY,JUMBO BAG RED RETROSPOT & JUMBO STORAGE BAG SUKI,0.0601,0.0372,0.0200,0.3328,8.9467,1.0,0.0178,1.4430,0.9451,0.2588,0.3070,0.4355


In [87]:
final_rules.describe()

,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
count,60.000000,60.000000,60.00000,60.000000,60.000000,60.0,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000
mean,0.073872,0.073872,0.02450,0.399318,5.778120,1.0,0.019540,1.720492,0.859290,0.207130,0.354700,0.399313
std,0.031412,0.031412,0.00546,0.180171,1.960348,0.0,0.005235,0.654247,0.107246,0.053241,0.183945,0.081499
min,0.026500,0.026500,0.02000,0.184300,1.878600,1.0,0.010600,1.116700,0.523300,0.114900,0.104500,0.206400
25%,0.043925,0.043925,0.02130,0.214425,4.604200,1.0,0.017000,1.227200,0.847700,0.168100,0.185075,0.346600
50%,0.063050,0.063050,0.02190,0.344950,5.808000,1.0,0.018500,1.443800,0.886050,0.195550,0.307400,0.425200
75%,0.106300,0.106300,0.02720,0.572050,7.018000,1.0,0.021700,2.136550,0.926850,0.258800,0.531950,0.453200
max,0.113400,0.113400,0.04220,0.801600,9.445100,1.0,0.035600,4.503400,0.970500,0.333900,0.777900,0.536800


In [88]:
final_rules.to_csv("top 10 product with highest total quantity rules.csv", index=False)

In [89]:
# 1 lấy top 10 sản phẩm doanh thu cao nhất
query = """
SELECT 
    p.item_name,
    SUM(d.quantity * d.price) AS revenue
FROM invoicedetail d
JOIN product p
    ON d.product_id = p.product_id
GROUP BY p.item_name
ORDER BY revenue DESC
LIMIT 10
"""

top_products = pd.read_sql(query, conn)

top10 = top_products["item_name"].tolist()

print("Top 10 products by revenue")
print(top_products)

# 3 tìm 2 rule lift cao nhất cho mỗi sản phẩm
result = []

for product in top10:

    related_rules = rules[
        rules["antecedents"].str.contains(product) |
        rules["consequents"].str.contains(product)
    ]

    top_rules = related_rules.sort_values("lift", ascending=False)

    top_rules["main_product"] = product

    result.append(top_rules)

final_rules1 = pd.concat(result)

final_rules1

C:\Users\ASUS\AppData\Local\Temp\ipykernel_16292\1596630176.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  top_products = pd.read_sql(query, conn)


Top 10 products by revenue
                            item_name    revenue
0            REGENCY CAKESTAND 3 TIER  165689.19
1  WHITE HANGING HEART T-LIGHT HOLDER  102588.37
2                       PARTY BUNTING   97367.48
3             JUMBO BAG RED RETROSPOT   93416.03
4                  RABBIT NIGHT LIGHT   66452.11
5      PAPER CHAIN KIT 50'S CHRISTMAS   63458.59
6       ASSORTED COLOUR BIRD ORNAMENT   58459.49
7                       CHILLI LIGHTS   53471.36
8             JUMBO BAG PINK POLKADOT   41829.94
9            BLACK RECORD COVER FRAME   40637.13


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,main_product
26,frozenset({'GREEN REGENCY TEACUP AND SAUCER'}),frozenset({'REGENCY CAKESTAND 3 TIER'}),0.0502,0.0982,0.0248,0.4943,5.0330,1.0,0.0199,1.7834,0.8437,0.2008,0.4393,0.3735,REGENCY CAKESTAND 3 TIER
27,frozenset({'REGENCY CAKESTAND 3 TIER'}),frozenset({'GREEN REGENCY TEACUP AND SAUCER'}),0.0982,0.0502,0.0248,0.2526,5.0330,1.0,0.0199,1.2709,0.8886,0.2008,0.2131,0.3735,REGENCY CAKESTAND 3 TIER
158,frozenset({'ROSES REGENCY TEACUP AND SAUCER'}),frozenset({'REGENCY CAKESTAND 3 TIER'}),0.0522,0.0982,0.0252,0.4832,4.9196,1.0,0.0201,1.7449,0.8406,0.2015,0.4269,0.3700,REGENCY CAKESTAND 3 TIER
159,frozenset({'REGENCY CAKESTAND 3 TIER'}),frozenset({'ROSES REGENCY TEACUP AND SAUCER'}),0.0982,0.0522,0.0252,0.2568,4.9196,1.0,0.0201,1.2753,0.8835,0.2015,0.2159,0.3700,REGENCY CAKESTAND 3 TIER
153,frozenset({'RED HANGING HEART T-LIGHT HOLDER'}),frozenset({'WHITE HANGING HEART T-LIGHT HOLDER'}),0.0372,0.1134,0.0247,0.6644,5.8592,1.0,0.0205,2.6415,0.8614,0.1963,0.6214,0.4411,WHITE HANGING HEART T-LIGHT HOLDER
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,frozenset({'JUMBO STORAGE BAG SUKI'}),frozenset({'JUMBO BAG PINK POLKADOT'}),0.0608,0.0625,0.0265,0.4363,6.9846,1.0,0.0227,1.6633,0.9123,0.2741,0.3988,0.4304,JUMBO BAG PINK POLKADOT
55,frozenset({'JUMBO SHOPPER VINTAGE RED PAISLEY'}),frozenset({'JUMBO BAG PINK POLKADOT'}),0.0601,0.0625,0.0241,0.4005,6.4112,1.0,0.0203,1.5639,0.8980,0.2445,0.3606,0.3931,JUMBO BAG PINK POLKADOT
54,frozenset({'JUMBO BAG PINK POLKADOT'}),frozenset({'JUMBO SHOPPER VINTAGE RED PAISLEY'}),0.0625,0.0601,0.0241,0.3856,6.4112,1.0,0.0203,1.5298,0.9003,0.2445,0.3463,0.3931,JUMBO BAG PINK POLKADOT
50,frozenset({'JUMBO BAG PINK POLKADOT'}),frozenset({'JUMBO BAG RED RETROSPOT'}),0.0625,0.1063,0.0422,0.6763,6.3610,1.0,0.0356,2.7608,0.8990,0.3339,0.6378,0.5368,JUMBO BAG PINK POLKADOT


In [91]:
# bỏ frozenset
final_rules1["antecedents"] = final_rules1["antecedents"].str.replace("frozenset({", "", regex=False)
final_rules1["antecedents"] = final_rules1["antecedents"].str.replace("})", "", regex=False)

final_rules1["consequents"] = final_rules1["consequents"].str.replace("frozenset({", "", regex=False)
final_rules1["consequents"] = final_rules1 ["consequents"].str.replace("})", "", regex=False)

# bỏ dấu nháy
final_rules1["antecedents"] = final_rules1["antecedents"].str.replace("'", "", regex=False)
final_rules1["consequents"] = final_rules1["consequents"].str.replace("'", "", regex=False)

# đổi dấu phẩy thành &
final_rules1["antecedents"] = final_rules1["antecedents"].str.replace(",", " &", regex=False)
final_rules1["consequents"] = final_rules1["consequents"].str.replace(",", " &", regex=False)

final_rules1.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,main_product
26,GREEN REGENCY TEACUP AND SAUCER,REGENCY CAKESTAND 3 TIER,0.0502,0.0982,0.0248,0.4943,5.0330,1.0,0.0199,1.7834,0.8437,0.2008,0.4393,0.3735,REGENCY CAKESTAND 3 TIER
27,REGENCY CAKESTAND 3 TIER,GREEN REGENCY TEACUP AND SAUCER,0.0982,0.0502,0.0248,0.2526,5.0330,1.0,0.0199,1.2709,0.8886,0.2008,0.2131,0.3735,REGENCY CAKESTAND 3 TIER
158,ROSES REGENCY TEACUP AND SAUCER,REGENCY CAKESTAND 3 TIER,0.0522,0.0982,0.0252,0.4832,4.9196,1.0,0.0201,1.7449,0.8406,0.2015,0.4269,0.3700,REGENCY CAKESTAND 3 TIER
159,REGENCY CAKESTAND 3 TIER,ROSES REGENCY TEACUP AND SAUCER,0.0982,0.0522,0.0252,0.2568,4.9196,1.0,0.0201,1.2753,0.8835,0.2015,0.2159,0.3700,REGENCY CAKESTAND 3 TIER
153,RED HANGING HEART T-LIGHT HOLDER,WHITE HANGING HEART T-LIGHT HOLDER,0.0372,0.1134,0.0247,0.6644,5.8592,1.0,0.0205,2.6415,0.8614,0.1963,0.6214,0.4411,WHITE HANGING HEART T-LIGHT HOLDER


In [92]:
final_rules1.to_csv("top 10 product with highest revenue rules.csv", index=False)

In [93]:
rules_sorted = rules.sort_values(by="lift", ascending=False)

In [94]:
rules_sorted.head(30)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
173,frozenset({'PINK REGENCY TEACUP AND SAUCER'}),"frozenset({'GREEN REGENCY TEACUP AND SAUCER', ...",0.0379,0.0377,0.0265,0.6993,18.5703,1.0,0.0251,3.2005,0.9834,0.5405,0.6876,0.7017
168,"frozenset({'GREEN REGENCY TEACUP AND SAUCER', ...",frozenset({'PINK REGENCY TEACUP AND SAUCER'}),0.0377,0.0379,0.0265,0.7041,18.5703,1.0,0.0251,3.2515,0.9832,0.5405,0.6924,0.7017
171,frozenset({'GREEN REGENCY TEACUP AND SAUCER'}),"frozenset({'ROSES REGENCY TEACUP AND SAUCER', ...",0.0502,0.0294,0.0265,0.5283,17.9972,1.0,0.0250,2.0576,0.9943,0.5000,0.5140,0.7158
170,"frozenset({'ROSES REGENCY TEACUP AND SAUCER', ...",frozenset({'GREEN REGENCY TEACUP AND SAUCER'}),0.0294,0.0502,0.0265,0.9033,17.9972,1.0,0.0250,9.8262,0.9730,0.5000,0.8982,0.7158
24,frozenset({'GREEN REGENCY TEACUP AND SAUCER'}),frozenset({'PINK REGENCY TEACUP AND SAUCER'}),0.0502,0.0379,0.0312,0.6208,16.3720,1.0,0.0293,2.5369,0.9885,0.5471,0.6058,0.7213
25,frozenset({'PINK REGENCY TEACUP AND SAUCER'}),frozenset({'GREEN REGENCY TEACUP AND SAUCER'}),0.0379,0.0502,0.0312,0.8218,16.3720,1.0,0.0293,5.3291,0.9759,0.5471,0.8123,0.7213
169,"frozenset({'GREEN REGENCY TEACUP AND SAUCER', ...",frozenset({'ROSES REGENCY TEACUP AND SAUCER'}),0.0312,0.0522,0.0265,0.8510,16.3009,1.0,0.0249,6.3608,0.9688,0.4664,0.8428,0.6794
172,frozenset({'ROSES REGENCY TEACUP AND SAUCER'}),"frozenset({'GREEN REGENCY TEACUP AND SAUCER', ...",0.0522,0.0312,0.0265,0.5079,16.3009,1.0,0.0249,1.9688,0.9904,0.4664,0.4921,0.6794
22,frozenset({'GARDENERS KNEELING PAD KEEP CALM'}),frozenset({'GARDENERS KNEELING PAD CUP OF TEA'}),0.0467,0.0389,0.0281,0.6011,15.4541,1.0,0.0262,2.4094,0.9811,0.4879,0.5850,0.6613
23,frozenset({'GARDENERS KNEELING PAD CUP OF TEA'}),frozenset({'GARDENERS KNEELING PAD KEEP CALM'}),0.0389,0.0467,0.0281,0.7215,15.4541,1.0,0.0262,3.4229,0.9731,0.4879,0.7078,0.6613


In [96]:
# sort rules theo lift
rules_sorted = rules.sort_values(by="lift", ascending=False)
rules_remove_index = pd.concat([final_rules, final_rules1]).index
rules_filtered = rules_sorted.drop(rules_remove_index)
rules_filtered.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
173,frozenset({'PINK REGENCY TEACUP AND SAUCER'}),"frozenset({'GREEN REGENCY TEACUP AND SAUCER', ...",0.0379,0.0377,0.0265,0.6993,18.5703,1.0,0.0251,3.2005,0.9834,0.5405,0.6876,0.7017
168,"frozenset({'GREEN REGENCY TEACUP AND SAUCER', ...",frozenset({'PINK REGENCY TEACUP AND SAUCER'}),0.0377,0.0379,0.0265,0.7041,18.5703,1.0,0.0251,3.2515,0.9832,0.5405,0.6924,0.7017
171,frozenset({'GREEN REGENCY TEACUP AND SAUCER'}),"frozenset({'ROSES REGENCY TEACUP AND SAUCER', ...",0.0502,0.0294,0.0265,0.5283,17.9972,1.0,0.0250,2.0576,0.9943,0.5000,0.5140,0.7158
170,"frozenset({'ROSES REGENCY TEACUP AND SAUCER', ...",frozenset({'GREEN REGENCY TEACUP AND SAUCER'}),0.0294,0.0502,0.0265,0.9033,17.9972,1.0,0.0250,9.8262,0.9730,0.5000,0.8982,0.7158
24,frozenset({'GREEN REGENCY TEACUP AND SAUCER'}),frozenset({'PINK REGENCY TEACUP AND SAUCER'}),0.0502,0.0379,0.0312,0.6208,16.3720,1.0,0.0293,2.5369,0.9885,0.5471,0.6058,0.7213


In [97]:
rules_filtered.describe()

,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
count,112.000000,112.000000,112.000000,112.000000,112.000000,112.0,112.000000,112.000000,112.000000,112.000000,112.000000,112.000000
mean,0.053373,0.053373,0.024918,0.485912,9.605459,1.0,0.022025,2.050060,0.932793,0.320264,0.456600,0.485909
std,0.011149,0.011149,0.003735,0.122674,3.514841,0.0,0.003636,1.054312,0.031596,0.093938,0.130409,0.104084
min,0.029400,0.029400,0.020100,0.253100,4.781700,1.0,0.015900,1.268000,0.835100,0.179100,0.211300,0.316600
25%,0.045100,0.045100,0.021625,0.404350,7.018100,1.0,0.019275,1.582400,0.910725,0.256700,0.368025,0.412075
50%,0.052900,0.052900,0.024800,0.459950,7.754100,1.0,0.021450,1.734100,0.923600,0.284700,0.423350,0.447500
75%,0.059200,0.059200,0.026800,0.543100,12.391275,1.0,0.024050,2.061325,0.960925,0.356900,0.514850,0.532825
max,0.079500,0.079500,0.037700,0.903300,18.570300,1.0,0.035000,9.826200,0.994300,0.581700,0.898200,0.735800


In [98]:
# bỏ frozenset
rules_filtered["antecedents"] = rules_filtered["antecedents"].str.replace("frozenset({", "", regex=False)
rules_filtered["antecedents"] = rules_filtered["antecedents"].str.replace("})", "", regex=False)

rules_filtered["consequents"] = rules_filtered["consequents"].str.replace("frozenset({", "", regex=False)
rules_filtered["consequents"] = rules_filtered["consequents"].str.replace("})", "", regex=False)

# bỏ dấu nháy
rules_filtered["antecedents"] = rules_filtered["antecedents"].str.replace("'", "", regex=False)
rules_filtered["consequents"] = rules_filtered["consequents"].str.replace("'", "", regex=False)

# đổi dấu phẩy thành &
rules_filtered["antecedents"] = rules_filtered["antecedents"].str.replace(",", " &", regex=False)
rules_filtered["consequents"] = rules_filtered["consequents"].str.replace(",", " &", regex=False)

rules_filtered.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
173,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER & ROSES REGENC...,0.0379,0.0377,0.0265,0.6993,18.5703,1.0,0.0251,3.2005,0.9834,0.5405,0.6876,0.7017
168,GREEN REGENCY TEACUP AND SAUCER & ROSES REGENC...,PINK REGENCY TEACUP AND SAUCER,0.0377,0.0379,0.0265,0.7041,18.5703,1.0,0.0251,3.2515,0.9832,0.5405,0.6924,0.7017
171,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER & PINK REGENCY...,0.0502,0.0294,0.0265,0.5283,17.9972,1.0,0.0250,2.0576,0.9943,0.5000,0.5140,0.7158
170,ROSES REGENCY TEACUP AND SAUCER & PINK REGENCY...,GREEN REGENCY TEACUP AND SAUCER,0.0294,0.0502,0.0265,0.9033,17.9972,1.0,0.0250,9.8262,0.9730,0.5000,0.8982,0.7158
24,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,0.0502,0.0379,0.0312,0.6208,16.3720,1.0,0.0293,2.5369,0.9885,0.5471,0.6058,0.7213


In [100]:
rules_filtered = rules_filtered.reset_index(drop=True)
rules_filtered["rule_name"] = "R" + (rules_filtered.index + 1).astype(str)


In [101]:
rules_filtered.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,rule_name
0,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER & ROSES REGENC...,0.0379,0.0377,0.0265,0.6993,18.5703,1.0,0.0251,3.2005,0.9834,0.5405,0.6876,0.7017,R1
1,GREEN REGENCY TEACUP AND SAUCER & ROSES REGENC...,PINK REGENCY TEACUP AND SAUCER,0.0377,0.0379,0.0265,0.7041,18.5703,1.0,0.0251,3.2515,0.9832,0.5405,0.6924,0.7017,R2
2,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER & PINK REGENCY...,0.0502,0.0294,0.0265,0.5283,17.9972,1.0,0.0250,2.0576,0.9943,0.5000,0.5140,0.7158,R3
3,ROSES REGENCY TEACUP AND SAUCER & PINK REGENCY...,GREEN REGENCY TEACUP AND SAUCER,0.0294,0.0502,0.0265,0.9033,17.9972,1.0,0.0250,9.8262,0.9730,0.5000,0.8982,0.7158,R4
4,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,0.0502,0.0379,0.0312,0.6208,16.3720,1.0,0.0293,2.5369,0.9885,0.5471,0.6058,0.7213,R5


In [102]:
rules_filtered.to_csv("normal product rules.csv", index=False)